# Pilot 5

- Upbringing (Good/Bad) × Action (Help/Harm)
- Blame (+1 to +7), Praise (+1 to +7), True Self (+1 to +7)
- N = 278

## Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Standard library imports
import re
import warnings

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########

# Measures
labels_measure = {
    "Blame or Praise": "Blame or Praise",
    "True Self":       "True Self"
}

# Gender
labels_gender = {
    1: 'Female',
    2: 'Male',
    3: 'Non-binary',
    4: 'Prefer not to say'
}


#################
# VISUALIZATION #
#################

# Colors
palette_main = {
    "Good": "#0072B2",
    "Bad":  "#D55E00"
}

## Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_5_data.csv', sep = ';')

# Define factors
factors_map = {
    'non,blameAc':         {'Upbringing': 'Good', 'Action': 'Harm (Blame)',  'Measure': 'Blame or Praise'},
    'non,blameSelf':       {'Upbringing': 'Good', 'Action': 'Harm (Blame)',  'Measure': 'True Self'},
    'non,praiseAc':        {'Upbringing': 'Good', 'Action': 'Help (Praise)', 'Measure': 'Blame or Praise'},
    'non,praiseSelf':      {'Upbringing': 'Good', 'Action': 'Help (Praise)', 'Measure': 'True Self'},
    'ignorant,blame,AC':   {'Upbringing': 'Bad',  'Action': 'Harm (Blame)',  'Measure': 'Blame or Praise'},
    'ignorant,blameSelf':  {'Upbringing': 'Bad',  'Action': 'Harm (Blame)',  'Measure': 'True Self'},
    'ignorant,praiseAc':   {'Upbringing': 'Bad',  'Action': 'Help (Praise)', 'Measure': 'Blame or Praise'},
    'ignorant,praiseSelf': {'Upbringing': 'Bad',  'Action': 'Help (Praise)', 'Measure': 'True Self'}
}

# Reshape wide to long
long_data = []

for _, row in df.iterrows():
    
    for col, factors in factors_map.items():
        score = pd.to_numeric(row.get(col), errors = 'coerce')
        
        if pd.notna(score):
            entry = {
                'ID':    row['ID'],
                'Score': score
            }
            
            entry.update(factors)
            long_data.append(entry)

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} observations).")

## Calculate Sociodemographics

In [ ]:
# Prepare variables
df['Gender_Labeled'] = df['Gender'].map(labels_gender)
df['Age'] = pd.to_numeric(df['Age'], errors = 'coerce')

# Calculate sociodemographics and display results
for col in ['Gender_Labeled']:
    print(f"\n{col}:")
    display(df[col].value_counts().to_frame('Frequency'))

print("\nAge:")
display(df['Age'].describe().to_frame('Value').round(3))

print("\nParticipants:")
total_participants = df_long['ID'].nunique()
display(f"N = {total_participants}")

## Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variable
group_factors = ['Measure', 'Action', 'Upbringing']
dependent_var = 'Score'

# Calculate descriptive statistics
descriptive_stats = (
    df_long
    .groupby(group_factors)[dependent_var]
    .agg(['mean', 'std', 'count'])
    .round(3)
)

# Display results
display(descriptive_stats)

## Perform ANOVAs

In [ ]:
for dv in df_long['Measure'].unique():
    print(f"\nANOVA ({dv})")
    subset = df_long[df_long['Measure'] == dv]
    
    # Define formula
    formula = "Score ~ C(Upbringing) * C(Action)"
    
    # Fit model
    model = ols(formula, data = subset).fit()
    
    # Run ANOVA
    aov_table = sm.stats.anova_lm(model, typ = 3)
    
    # Calculate effect sizes
    aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])
    
    # Display results
    display(aov_table.round(3))

## Perform t-Tests

In [ ]:
# Initialize lists
t_results = []
p_values = []

# Perform t-tests
for measure in df_long['Measure'].unique():
    for action in df_long['Action'].unique():
        
        good_scores = df_long[(df_long['Measure'] == measure) & (df_long['Action'] == action) & (df_long['Upbringing'] == 'Good')]['Score']
        bad_scores = df_long[(df_long['Measure'] == measure) & (df_long['Action'] == action) & (df_long['Upbringing'] == 'Bad')]['Score']
        
        t_stat, p_val = stats.ttest_ind(good_scores, bad_scores, equal_var = False)
        
        p_values.append(p_val)
        t_results.append({
            'DV':                   measure,
            'Condition':            action,
            'Mean Good Upbringing': round(good_scores.mean(), 3),
            'Mean Bad Upbringing':  round(bad_scores.mean(), 3),
            't':                    round(t_stat, 3),
            'p_uncorrected':        p_val
        })

# Apply Bonferroni correction
reject, pvals_corrected, _, _ = multipletests(p_values, alpha = 0.05, method = 'bonferroni')

for i, res in enumerate(t_results):
    res['p_bonferroni'] = round(pvals_corrected[i], 3)
    res['p_uncorrected'] = round(res['p_uncorrected'], 3)

# Display results
display(pd.DataFrame(t_results))

## Generate Histograms

In [ ]:
# Set condition labels
df_long['Condition'] = (df_long['Action'].astype(str) + " / " +
                        df_long['Upbringing'].astype(str)
                       )

# Generate graphs
for measure in df_long['Measure'].unique():
    subset = df_long[df_long['Measure'] == measure]
    
    g = sns.displot(data     = subset,
                    x        = "Score",
                    col      = "Condition",
                    col_wrap = 4,
                    kind     = "hist",
                    discrete = True,
                    shrink   = 0.8,
                    height   = 3,
                    aspect   = 1.2,
                    color    = "#0072B2"
                   )
    
    # Set axis labels and subplot titles
    g.set_axis_labels("Value", "Count")
    g.set_titles("{col_name}")
    
    # Limit x-axis
    for ax in g.axes.flat:
        ax.set_xticks(range(1, 8))
        ax.set_xlim(0.5, 7.5)
    
    # Add main title
    plt.subplots_adjust(top = 0.8)
    g.fig.suptitle(labels_measure.get(measure, measure), fontsize = 16)
    
    # Export graph
    clean_name = measure.replace(" ", "_").lower()
    filename = f'blame_praise_self_pilot_5_histograms_{clean_name}.png'
    g.savefig(filename, dpi = 300, bbox_inches = "tight")
    plt.show()
    print(f"Graph saved as '{filename}'.")

## Generate Bar Plots

In [ ]:
# Generate graphs
for measure in df_long['Measure'].unique():
    plt.figure(figsize = (8, 5))
    
    g = sns.barplot(data    = df_long[df_long['Measure'] == measure],
                    x       = "Action",
                    y       = "Score",
                    hue     = "Upbringing",
                    palette = palette_main,
                    capsize = .05,
                    ci      = 68
                   )
    
    # Set axis labels and subplot titles
    plt.ylabel('Mean Score (1 to 7)', fontsize = 12)
    plt.xlabel('Action', fontsize = 12)
    plt.ylim(1, 7.5)
    
    # Add main title
    plt.title(f'{measure}', fontsize = 14)
    
    # Add legend
    plt.legend(title = 'Upbringing', bbox_to_anchor = (1.05, 1), loc = 'upper left')
    
    # Export graph
    clean_name = measure.replace(" ", "_").lower()
    filename = f'blame_praise_self_pilot_5_bar_plot_{clean_name}.png'
    plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    print(f"Graph saved as '{filename}'.")